# spaCy y los desafíos del español rioplatense

**Duración estimada:** 30 minutos

## Propósito
Los modelos estadísticos de spaCy fueron entrenados principalmente sobre corpus del español peninsular o un registro neutro internacional (AnCora). En este cuaderno vas a observar de primera mano cómo reacciona el modelo frente a particularidades de nuestra variante regional y vas a aprender a corregir esos sesgos con herramientas basadas en reglas.

## Relación con los cuadernos anteriores
- En `001` a `003` trabajaste con el pipeline estándar de spaCy: tokenización, lematización, POS, NER y dependencias.
- Acá vas a diagnosticar dónde ese pipeline falla en contexto argentino y a intervenir con componentes que lo complementan.

## Resultados de aprendizaje
Al finalizar este cuaderno vas a poder:
- diagnosticar errores concretos de lematización y etiquetado POS frente al voseo;
- utilizar `Matcher` para capturar locuciones y construcciones que el modelo estadístico no reconoce;
- configurar `EntityRuler` para priorizar diccionarios locales por sobre las decisiones probabilísticas del NER estándar.

In [ ]:
!pip install spacy -q
!python -m spacy download es_core_news_sm -q

In [ ]:
import spacy
from spacy.matcher import Matcher

nlp = spacy.load("es_core_news_sm")
print(f"Modelo cargado: {nlp.meta['name']}")
print(f"Pipeline activo: {nlp.pipe_names}")

---
## 1. El diagnóstico: dónde falla el modelo con el voseo

Vamos a contrastar frases equivalentes en español neutro y en español rioplatense. El objetivo no es "romper" el modelo, sino entender exactamente qué tipo de errores produce y por qué impactan en un análisis posterior.

### 1.1 Verbos conjugados: *tenés*, *sabés*, *sos*

Observá cómo el modelo asigna lema y categoría gramatical (POS) a las mismas ideas expresadas en dos variantes.

In [ ]:
oraciones = [
    ("Tú tienes razón y sabes lo que hacés.",
     "Vos tenés razón y sabés lo que hacés."),
    ("Tú eres mi colega.",
     "Vos sos mi colega."),
]

for neutro_txt, voseo_txt in oraciones:
    doc_n = nlp(neutro_txt)
    doc_v = nlp(voseo_txt)

    print(f"\nNEUTRO: {neutro_txt}")
    for t in doc_n:
        if not t.is_punct and not t.is_space:
            print(f"  {t.text:15} -> lema={t.lemma_:15} POS={t.pos_}")

    print(f"VOSEO:  {voseo_txt}")
    for t in doc_v:
        if not t.is_punct and not t.is_space:
            print(f"  {t.text:15} -> lema={t.lemma_:15} POS={t.pos_}")

> **Lectura de los resultados:**
>
> Observá las filas de *tenés*, *sabés* y *sos* en la columna voseo.
> - `tenés` recibe lema `tenés` y POS `ADJ` — el modelo no reconoce que es el verbo *tener* conjugado en segunda persona del voseo.
> - `sabés` recibe lema `sabé` y POS `NOUN` — ni siquiera lo clasifica como verbo.
> - `sos` recibe lema `sos` y POS `PROPN` — lo interpreta como un nombre propio, no como el verbo *ser*.
> - `Vos` también aparece como `PROPN` en lugar de `PRON`.
>
> Compará con la columna neutra, donde `tienes` → `tener`/`VERB` y `eres` → `ser`/`AUX` funcionan sin error.

### 1.2 Imperativo con voseo: *revisá*, *analizá*, *compartí*

Una instrucción tan habitual como "Revisá los datos" puede quedar completamente invisible para el pipeline si el modelo interpreta los imperativos del voseo como nombres propios.

In [ ]:
imperativo_vos = nlp("Revisá los datos, analizá las tendencias y compartí tus conclusiones.")
imperativo_tu  = nlp("Revisa los datos, analiza las tendencias y comparte tus conclusiones.")

print("IMPERATIVO VOSEO:")
for t in imperativo_vos:
    if not t.is_punct and not t.is_space:
        print(f"  {t.text:15} -> lema={t.lemma_:15} POS={t.pos_}")

print("\nIMPERATIVO NEUTRO:")
for t in imperativo_tu:
    if not t.is_punct and not t.is_space:
        print(f"  {t.text:15} -> lema={t.lemma_:15} POS={t.pos_}")

> **Impacto concreto:**
> Si estás contando verbos frecuentes en un corpus de noticias argentinas (como vas a hacer en el Laboratorio Integrador), todos los imperativos del voseo quedarán fuera del conteo. Esto introduce un sesgo silencioso en los resultados.

---
## 2. Enfoque híbrido: capturando locuciones con `Matcher`

Cuando el modelo estadístico no reconoce una construcción, no lo descartamos: lo complementamos con reglas explícitas. El componente `Matcher` de spaCy permite definir patrones a nivel de token para capturar secuencias que el modelo ignora.

En el periodismo argentino aparecen con frecuencia locuciones verbales y temporales que el modelo no agrupa como unidad. Vamos a capturarlas.

In [ ]:
texto = nlp(
    "La puesta en marcha del proyecto busca poner en valor "
    "la producción regional a corto plazo."
)

matcher = Matcher(nlp.vocab)

# Patrón 1: Locución verbal 'puesta en marcha'
matcher.add("LOCUCION_VERBAL", [
    [{"LOWER": "puesta"}, {"LOWER": "en"}, {"LOWER": "marcha"}],
    [{"LOWER": "poner"}, {"LOWER": "en"}, {"LOWER": "valor"}],
])

# Patrón 2: Locución temporal 'a corto plazo'
matcher.add("LOCUCION_TEMPORAL", [
    [{"LOWER": "a"}, {"LOWER": "corto"}, {"LOWER": "plazo"}],
])

resultados = matcher(texto)

print("Locuciones detectadas con Matcher:")
for match_id, inicio, fin in resultados:
    etiqueta = nlp.vocab.strings[match_id]
    span = texto[inicio:fin]
    print(f"  {etiqueta}: '{span.text}'")

El `Matcher` opera sobre atributos de cada token (texto, lema, POS, forma, etc.) y permite combinar condiciones con operadores de cantidad (`OP`). Esto lo convierte en una herramienta precisa para capturar estructuras multi-palabra que un modelo estadístico trata como tokens independientes.

---
## 3. Priorizando entidades locales: `EntityRuler`

El componente de reconocimiento de entidades nombradas (`ner`) puede ignorar organismos argentinos, asignarles una categoría incorrecta o incluso capturar fragmentos de más. Vamos a verificarlo con un ejemplo real y después corregirlo inyectando un `EntityRuler` en el pipeline.

### 3.1 Línea base: el NER estándar solo

In [ ]:
texto_noticia = (
    "La ANSES confirmó aumentos para jubilados. "
    "El BCRA mantendrá la tasa de referencia. "
    "La CGT convocó a una reunión en CABA "
    "con funcionarios del Ministerio de Trabajo."
)

doc_base = nlp(texto_noticia)

print("--- Entidades detectadas con el NER estándar ---")
for ent in doc_base.ents:
    print(f"  {ent.text:30} -> {ent.label_} ({spacy.explain(ent.label_)})")

> **Problemas observables:**
> - `El BCRA mantendrá` se captura como una sola entidad de tipo `MISC` — el modelo no logró delimitar correctamente la entidad y arrastró tokens de más.
> - `Ministerio de Trabajo` se clasifica como `LOC` (lugar) en lugar de `ORG` (organización).
> - Según la variante de escritura, `ANSES` / `ANSeS` puede no ser detectada en absoluto.
>
> Este tipo de errores se propaga al dashboard y a los conteos del Laboratorio Integrador si no se interviene.

### 3.2 Corrección: inyectando un `EntityRuler` antes del NER

El `EntityRuler` permite definir un diccionario de entidades que se evalúa *antes* de que actúe el modelo estadístico. Si hay conflicto, nuestras reglas tienen prioridad.

In [ ]:
# Cargamos un modelo limpio (sin modificaciones previas)
nlp_mejorado = spacy.load("es_core_news_sm")

# Creamos el EntityRuler y lo insertamos ANTES del componente 'ner'
config = {"overwrite_ents": True}
ruler = nlp_mejorado.add_pipe("entity_ruler", config=config, before="ner")

# Diccionario de entidades locales
patrones_locales = [
    # Organismos gubernamentales
    {"label": "ORG", "pattern": "ANSES"},
    {"label": "ORG", "pattern": "ANSeS"},
    {"label": "ORG", "pattern": "BCRA"},
    {"label": "ORG", "pattern": "CGT"},
    {"label": "ORG", "pattern": "AFIP"},
    {"label": "ORG", "pattern": "CONICET"},
    # Ubicaciones
    {"label": "LOC", "pattern": "CABA"},
    {"label": "LOC", "pattern": "AMBA"},
    # Patrones multi-token
    {"label": "ORG", "pattern": "Casa Rosada"},
    {"label": "ORG", "pattern": [{"LOWER": "ministerio"}, {"LOWER": "de"}, {"LOWER": "trabajo"}]},
]

ruler.add_patterns(patrones_locales)
print(f"Pipeline modificado: {nlp_mejorado.pipe_names}")

In [ ]:
# Reprocesamos el MISMO texto con el pipeline mejorado
doc_mejorado = nlp_mejorado(texto_noticia)

print("--- Entidades detectadas con EntityRuler + NER ---")
for ent in doc_mejorado.ents:
    print(f"  {ent.text:30} -> {ent.label_} ({spacy.explain(ent.label_)})")

> **Comparación:**
> - `BCRA` ahora se delimita correctamente y se clasifica como `ORG`.
> - `Ministerio de Trabajo` pasa de `LOC` a `ORG`.
> - El `EntityRuler` no interfiere con las entidades que el modelo ya clasificaba bien (`CGT`, `CABA`).

---
## Cierre y preparación para el Laboratorio Integrador

En este cuaderno observaste tres problemas concretos del procesamiento de textos en español rioplatense:
1. **Lematización rota** — verbos conjugados con voseo reciben lemas incorrectos o directamente se clasifican como sustantivos o adjetivos.
2. **Locuciones fragmentadas** — expresiones multi-palabra como *puesta en marcha* no se agrupan a menos que intervengamos con `Matcher`.
3. **Entidades mal clasificadas o ausentes** — organismos argentinos reciben etiquetas incorrectas o se pierden si el modelo no los conoce.

Estos tres problemas impactan directamente en la calidad de los datos que vas a procesar en el `005_laboratorio_integrador_noticias` y en el `TPI_1`. No se busca que corrijas a mano cada excepción, pero **sí se evalúa** que en tu *AI Reflection Log* dejes constancia de cuándo el modelo falló y por qué. Mantener ese juicio analítico activo es la competencia central.